In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q transformers datasets bitsandbytes trl peft huggingface_hub

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_DIR = "/content/drive/MyDrive/finetuning"
else:
    BASE_DIR = "."

BASE_DIR

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/shankuriakose/finetuning.git"
    REPO_DIR = "/content/finetuning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}

    %cd {REPO_DIR}
    sys.path.insert(0, REPO_DIR)

In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [1]:

import torch

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
device

device(type='mps')

In [ ]:
from pathlib import Path

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from download import download_model


def load_model_and_tokenizer(model_name: str, base_dir: Path | str = "."):
    local_dir = download_model(model_name, base_dir=base_dir)

    config_8bit = BitsAndBytesConfig(load_in_8bit=True)

    model_8bit = AutoModelForCausalLM.from_pretrained(
        local_dir,
        quantization_config=config_8bit,
        device_map="auto",
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(local_dir, trust_remote_code=True, padding_side="left")

    return model_8bit, tokenizer


model_name = "meta-llama/Llama-3.2-1B-Instruct"
model_8bit, tokenizer = load_model_and_tokenizer(model_name, base_dir=BASE_DIR)